# Custom Transfer Learning for the Universal Brain Encoder

This notebook adapts a **pretrained Universal Brain Encoder** to a new subject (or dataset) using your own **image ↔ fMRI** pairs.

It mirrors `train_encoder_transfer_custom.py` and adds **inference** and **evaluation**.

## What transfer learning does here

The encoder maps images → predicted fMRI responses. Most of the network is shared across brains; what is **subject-specific** is the `voxel_embed` table (one embedding vector per voxel).

Transfer procedure:
1. Load a pretrained encoder checkpoint.
2. **Freeze** all weights.
3. **Replace** `voxel_embed` with a new randomly initialized table sized to your subject's voxel count `V`.
4. Fine-tune only that table on your image/fMRI pairs.

## Expected data format

| Array | Shape | Dtype | Notes |
|---|---|---|---|
| Images | `[N, H, W, 3]` | `uint8` | typically `H=W=224` |
| fMRI | `[N, V]` | float | **subject-local** betas, one row per image |

Optional validation arrays use the same shapes.

## Notebook outline

1. Configuration
2. Load data & build dataloaders
3. Initialize / transfer the model
4. Train
5. Inference (predict fMRI for images)
6. Evaluation (per-voxel Pearson correlations)

## 1. Configuration

Set paths to your `.npy` files and training hyperparameters below.

- If `VAL_IMAGES` / `VAL_FMRI` are `None`, training still runs and the **last-epoch** checkpoint is saved (no early selection by validation).
- `BASE_MODEL` should point to a pretrained encoder (e.g. the Hugging Face checkpoint under `results/saved_models/`).

In [ ]:
import os
import sys

# Repo root (parent of train_transfer/)
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if os.path.basename(os.getcwd()) != "train_transfer":
    # Allow running from repo root as well
    REPO_ROOT = os.getcwd() if os.path.isdir(os.path.join(os.getcwd(), "train_transfer")) else REPO_ROOT
os.chdir(REPO_ROOT)
sys.path.append(REPO_ROOT)
print("Working directory:", REPO_ROOT)

# -------------------- user config --------------------
GPU = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU

TRAIN_IMAGES = "path/to/train_images.npy"   # [N, H, W, 3] uint8
TRAIN_FMRI   = "path/to/train_fmri.npy"     # [N, V] float
VAL_IMAGES   = "path/to/val_images.npy"     # optional; set None to skip val
VAL_FMRI     = "path/to/val_fmri.npy"       # optional; set None to skip val

BASE_MODEL = f"results/saved_models/encoder_ch128.pth"
NAME = "encoder_transfer_custom"
SAVE_PATH = f"results/saved_models/transfer/{NAME}.pth"
OUTPUT_DIR = f"results/encoder_predictions/{NAME}/"

NUM_SAMPLES = None   # e.g. 1000 to cap training samples; None = use all
EPOCHS = 50
BATCH_SIZE = 32
LR = 1e-3

DROPOUT = 0.25
EMBED_DIM_VOX = 256
INNER_CH = 128
# -----------------------------------------------------

## 2. Imports

In [ ]:
import numpy as np
import torch
import torch.optim as optim
from torch.utils import data
from tensorboardX import SummaryWriter
from scipy.ndimage import shift
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
from pathlib import Path

from utils.train_utils_enc import train, test
from models.encoder_models import encoder_param
from utils.datasets import EncDataset

torch.set_default_tensor_type("torch.FloatTensor")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 3. Load image / fMRI arrays

Training uses **paired** samples: image `i` must correspond to fMRI row `i`.

Subject indices are all zeros because this custom pipeline trains a **single** subject embedding table of size `V`.

In [ ]:
print(f"Loading train images from {TRAIN_IMAGES}")
images_train = np.load(TRAIN_IMAGES)
print(f"Loading train fMRI from {TRAIN_FMRI}")
fmri_train = np.load(TRAIN_FMRI).astype(np.float32)

assert images_train.shape[0] == fmri_train.shape[0], "Train images and fMRI must have the same N"

if NUM_SAMPLES is not None:
    images_train = images_train[:NUM_SAMPLES]
    fmri_train = fmri_train[:NUM_SAMPLES]

sub_train = np.zeros(fmri_train.shape[0], dtype=int)
num_voxels_subjects = np.array([fmri_train.shape[1]])
NUM_VOXELS = int(num_voxels_subjects.sum())

use_val = VAL_IMAGES is not None and VAL_FMRI is not None
if use_val:
    print(f"Loading val images from {VAL_IMAGES}")
    images_val = np.load(VAL_IMAGES)
    print(f"Loading val fMRI from {VAL_FMRI}")
    fmri_val = np.load(VAL_FMRI).astype(np.float32)
    assert images_val.shape[0] == fmri_val.shape[0]
    assert fmri_val.shape[1] == NUM_VOXELS, "Val voxel count must match train"
    sub_val = np.zeros(fmri_val.shape[0], dtype=int)
else:
    images_val = fmri_val = sub_val = None

print(f"Train: {images_train.shape[0]} samples, images {images_train.shape[1:]}, voxels V={NUM_VOXELS}")
if use_val:
    print(f"Val:   {images_val.shape[0]} samples")
else:
    print("Val:   none (will save last-epoch checkpoint)")

## 4. Preprocessing & dataloaders

- Images are scaled to `[0, 1]`, then ImageNet-normalized, then transposed to `CHW` for PyTorch.
- **Training** uses a small random spatial shift (`max_shift=3`) as data augmentation.
- **Validation / inference** use the deterministic transform (no shift).
- During training, `EncDataset` randomly samples `5000` voxels per batch item for efficiency; validation uses all voxels.

In [ ]:
mean = np.array([0.485, 0.456, 0.406]).reshape([1, 1, 3]).astype(float)
std = np.array([0.229, 0.224, 0.225]).reshape([1, 1, 3]).astype(float)


def trans_imgs(imgs):
    """Deterministic preprocess for val / inference."""
    imgs = imgs / 255.0
    imgs = (imgs - mean) / std
    imgs = imgs.transpose([2, 0, 1])
    return torch.from_numpy(imgs.astype(float)).float()


def rand_shift(img, max_shift=0):
    x_shift, y_shift = np.random.randint(-max_shift, max_shift + 1, size=2)
    return shift(img, [x_shift, y_shift, 0], prefilter=False, order=0, mode="nearest")


def trans_imgs_shift(img, max_shift=3):
    """Train preprocess with random shift augmentation."""
    img = img / 255.0
    img = rand_shift(img, max_shift)
    img = (img - mean) / std
    img = img.transpose([2, 0, 1])
    return torch.from_numpy(img.astype(float)).float()


train_loader = data.DataLoader(
    EncDataset(
        images_train,
        fmri_train,
        sub_train,
        num_voxels_subjects,
        preprocess=trans_imgs_shift,
        num_voxels_to_sample=5000,
    ),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
)

val_loader = None
if use_val:
    val_loader = data.DataLoader(
        EncDataset(
            images_val,
            fmri_val,
            sub_val,
            num_voxels_subjects,
            sample=False,
            preprocess=trans_imgs,
        ),
        batch_size=8,
        shuffle=False,
        num_workers=1,
    )

print("Train batches:", len(train_loader))
if val_loader is not None:
    print("Val batches:", len(val_loader))

## 5. Initialize the transfer model

1. Load the pretrained encoder.
2. Freeze every parameter.
3. Allocate a new `voxel_embed` of shape `[V, embed_dim]` and train only that.

The initialization scale matches the original encoder setup (`enc_param.init / (2 * sqrt(embed_dim))`).

In [ ]:
enc_param = encoder_param(NUM_VOXELS)
enc_param.inner_ch = INNER_CH
enc_param.drop_out = DROPOUT
enc_param.embed_dim_vox = EMBED_DIM_VOX
enc_param.in_spatial = 257

print(f"Loading base model from {BASE_MODEL}")
model = torch.load(BASE_MODEL, map_location=device)

for param in model.parameters():
    param.requires_grad = False

model.voxel_embed = torch.nn.Parameter(
    (enc_param.init / (2 * np.sqrt(enc_param.embed_dim_vox)))
    * torch.randn(NUM_VOXELS, enc_param.embed_dim_vox),
    requires_grad=True,
).float()

model = model.to(device)
trainable = [n for n, p in model.named_parameters() if p.requires_grad]
print("Trainable parameters:", trainable)
print("voxel_embed shape:", tuple(model.voxel_embed.shape))

## 6. Train

The training / validation helpers (`train`, `test` from `utils.train_utils_enc`) log MSE, MAE, and correlation percentiles to TensorBoard under `logs/tensorboard/encoder_exp/<NAME>`.

Loss used during training:

$$\mathcal{L} = \mathrm{MSE}(\hat{y}, y) - 0.1 \cdot \mathrm{mean}(\mathrm{cosine\_sim}(\hat{y}, y))$$

Checkpointing:
- **With validation:** save whenever the validation 75th-percentile voxel correlation improves.
- **Without validation:** overwrite the checkpoint every epoch (last epoch kept).

In [ ]:
os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
writer = SummaryWriter("logs/tensorboard/encoder_exp/" + NAME)
optimizer = optim.Adam(model.parameters(), lr=LR, amsgrad=True)

best_metric = 0.0
history = {"epoch": [], "val_corr_p75": []}

for epoch in range(1, EPOCHS + 1):
    print(f"\n===== Epoch {epoch}/{EPOCHS} =====")
    train(model, device, train_loader, optimizer, epoch, writer)

    if use_val:
        metric = test(model, device, val_loader, epoch, writer)
        history["epoch"].append(epoch)
        history["val_corr_p75"].append(metric)
        if metric > best_metric:
            best_metric = metric
            torch.save(model, SAVE_PATH)
            print(f"Saved best model (val corr p75={best_metric:.4f}) -> {SAVE_PATH}")
    else:
        torch.save(model, SAVE_PATH)

if not use_val:
    print(f"Model saved after {EPOCHS} epochs -> {SAVE_PATH}")
else:
    print(f"Best val corr p75: {best_metric:.4f}")

writer.close()

In [ ]:
# Optional: plot validation metric over epochs
if use_val and len(history["epoch"]) > 0:
    plt.figure(figsize=(6, 3.5))
    plt.plot(history["epoch"], history["val_corr_p75"], marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Val voxel corr (75th percentile)")
    plt.title("Transfer training progress")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No validation history to plot.")

## 7. Inference

Reload the saved transfer checkpoint and predict full fMRI responses for a set of images.

By default we predict on the **validation** set (if provided); otherwise we fall back to the training images.

For a custom transfer model there is a single subject embedding table, so voxel indices are simply `0 … V-1`.

In [ ]:
def predict_fmri(encoder_model, images, num_voxels, device, batch_hint=100):
    """Predict [N, V] fMRI for a single-subject transfer encoder."""
    encoder_model.eval()
    n_images = images.shape[0]
    vox_ind = torch.arange(num_voxels, device=device).unsqueeze(0)
    preds = np.zeros([n_images, num_voxels], dtype=np.float32)

    for i in range(n_images):
        if i % batch_hint == 0:
            print(f"  Processing image {i}/{n_images}")
        x = trans_imgs(images[i]).unsqueeze(0).to(device)
        with torch.no_grad():
            pred = encoder_model(x, vox_ind)
        preds[i] = pred.detach().cpu().numpy()
    return preds


# Choose evaluation images / ground truth
if use_val:
    eval_images, eval_fmri, eval_split = images_val, fmri_val, "val"
else:
    eval_images, eval_fmri, eval_split = images_train, fmri_train, "train"
    print("Warning: no val set — evaluating on training data (optimistic).")

print(f"Loading transfer model from {SAVE_PATH}")
encoder = torch.load(SAVE_PATH, map_location=device).eval().to(device)

print(f"Predicting fMRI on {eval_split} ({eval_images.shape[0]} images, V={NUM_VOXELS})...")
pred_fmri = predict_fmri(encoder, eval_images, NUM_VOXELS, device)
print("Predictions shape:", pred_fmri.shape)

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
pred_path = out_dir / f"pred_fmri_{eval_split}.npz"
np.savez(pred_path, subject_custom=pred_fmri.astype(np.float16))
print(f"Saved predictions -> {pred_path}")

## 8. Evaluation

For each voxel $v$, compute Pearson correlation between predicted and measured responses across images:

$$
r_v = \mathrm{corr}\big(\hat{y}_{:,v},\, y_{:,v}\big)
$$

We then summarize the distribution of $\{r_v\}$ (median / percentiles) and plot a histogram.

Higher median / p75 correlation ⇒ better encoding accuracy on held-out images.

In [ ]:
def compute_voxel_correlations(y_true, y_pred):
    num_voxels = y_true.shape[1]
    corr = np.zeros(num_voxels, dtype=np.float32)
    for i in range(num_voxels):
        corr[i] = pearsonr(y_true[:, i], y_pred[:, i])[0]
    return np.nan_to_num(corr)


voxel_corr = compute_voxel_correlations(eval_fmri, pred_fmri)

summary = {
    "mean": float(np.mean(voxel_corr)),
    "median": float(np.median(voxel_corr)),
    "p75": float(np.percentile(voxel_corr, 75)),
    "p90": float(np.percentile(voxel_corr, 90)),
}
print(f"Voxel correlation summary ({eval_split}):")
for k, v in summary.items():
    print(f"  {k:>6}: {v:.4f}")

corr_path = out_dir / f"voxel_corr_{eval_split}.npz"
np.savez(corr_path, subject_custom=voxel_corr, **{f"summary_{k}": np.array(v) for k, v in summary.items()})
print(f"Saved correlations -> {corr_path}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))

axes[0].hist(voxel_corr, bins=50, color="#3d5a80", edgecolor="white")
axes[0].axvline(summary["median"], color="#e07a5f", linestyle="--", label=f"median={summary['median']:.3f}")
axes[0].axvline(summary["p75"], color="#81b29a", linestyle="--", label=f"p75={summary['p75']:.3f}")
axes[0].set_xlabel("Pearson r")
axes[0].set_ylabel("# voxels")
axes[0].set_title(f"Per-voxel encoding accuracy ({eval_split})")
axes[0].legend()

# Example: predicted vs true for the best voxel
best_v = int(np.argmax(voxel_corr))
axes[1].scatter(eval_fmri[:, best_v], pred_fmri[:, best_v], s=10, alpha=0.5, color="#3d5a80")
lims = [
    min(eval_fmri[:, best_v].min(), pred_fmri[:, best_v].min()),
    max(eval_fmri[:, best_v].max(), pred_fmri[:, best_v].max()),
]
axes[1].plot(lims, lims, "--", color="#e07a5f", linewidth=1)
axes[1].set_xlabel("Measured fMRI")
axes[1].set_ylabel("Predicted fMRI")
axes[1].set_title(f"Best voxel (idx={best_v}, r={voxel_corr[best_v]:.3f})")

plt.tight_layout()
plt.show()

## 9. Predict on new images (optional)

Use the fine-tuned model on any new `[N, H, W, 3]` uint8 image array — no ground-truth fMRI required.

In [ ]:
# Set a path to run; leave as None to skip
NEW_IMAGES = None  # e.g. "path/to/new_images.npy"

if NEW_IMAGES is not None:
    new_images = np.load(NEW_IMAGES)
    print(f"Predicting for {new_images.shape[0]} new images...")
    new_pred = predict_fmri(encoder, new_images, NUM_VOXELS, device)
    new_path = out_dir / "pred_fmri_new.npz"
    np.savez(new_path, subject_custom=new_pred.astype(np.float16))
    print(f"Saved -> {new_path}  shape={new_pred.shape}")
else:
    print("Set NEW_IMAGES to a .npy path to run prediction on new images.")

## Notes & tips

- **CLI equivalent:** `python train_transfer/train_encoder_transfer_custom.py --images ... --fmri ...`
- Prefer a held-out validation set for model selection; evaluating on train alone overestimates performance.
- Images should match the encoder's expected resolution (NSD pipeline uses 224×224).
- fMRI should be **subject-local** (shape `[N, V]` for that subject only), not concatenated multi-subject space.
- TensorBoard logs: `tensorboard --logdir logs/tensorboard/encoder_exp`